## Before you start

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [1]:
!nvidia-smi

Wed Oct 29 03:01:20 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
HOME = os.getcwd()
print(HOME)

/content


## Install Grounding DINO

---



In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set the file path
zip_path = "/content/drive/MyDrive/Agri-VG.zip"
extract_to = "/content"

# Unzip the file
import zipfile
import os

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("✅ Extraction complete!")
print("Files extracted to:", os.path.abspath(extract_to))


Mounted at /content/drive
✅ Extraction complete!
Files extracted to: /content


In [4]:
%cd {HOME}
#!git clone https://github.com/IDEA-Research/GroundingDINO.git
%cd /content/Agri-VG/GroundingDINO/groundingdino/models/GroundingDINO/csrc/MsDeformAttn
!sed -i 's/value.type()/value.scalar_type()/g' ms_deform_attn_cuda.cu
!sed -i 's/value.scalar_type().is_cuda()/value.is_cuda()/g' ms_deform_attn_cuda.cu
%cd /content/Agri-VG/GroundingDINO
!pip install -q -e .
!pip install -q roboflow

/content
/content/Agri-VG/GroundingDINO/groundingdino/models/GroundingDINO/csrc/MsDeformAttn
/content/Agri-VG/GroundingDINO
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.2/207.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 117.5 MB/s eta 0:00:00


In [5]:
import os

#CONFIG_PATH = os.path.join(HOME, "/content/Agri-VG/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py")
CONFIG_PATH = os.path.join(HOME, "/content/Agri-VG/GroundingDINO/groundingdino/config/GroundingDINO_SwinB_cfg.py")
print(CONFIG_PATH, "; exist:", os.path.isfile(CONFIG_PATH))

/content/Agri-VG/GroundingDINO/groundingdino/config/GroundingDINO_SwinB_cfg.py ; exist: True


## Download Grounding DINO Weights 🏋️

In [6]:
%cd {HOME}
!mkdir {HOME}/weights
%cd {HOME}/weights

#!wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
!wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha2/groundingdino_swinb_cogcoor.pth

/content
/content/weights


In [7]:
import os

#WEIGHTS_NAME = "groundingdino_swint_ogc.pth"
WEIGHTS_NAME = "groundingdino_swinb_cogcoor.pth"

WEIGHTS_PATH = os.path.join(HOME, "weights", WEIGHTS_NAME)
print(WEIGHTS_PATH, "; exist:", os.path.isfile(WEIGHTS_PATH))

/content/weights/groundingdino_swinb_cogcoor.pth ; exist: True


In [8]:
cd /content/Agri-VG/GroundingDINO

/content/Agri-VG/GroundingDINO


In [9]:
try:
    from groundingdino.util.slconfig import SLConfig
    from groundingdino.models import build_model
    from groundingdino.hierarchical.annotation import load_gref_annotations
    from groundingdino.hierarchical.batching import UniqueNegSentenceBatchSampler
    from groundingdino.hierarchical.losses import HMLC
    from groundingdino.hierarchical.instance_query_init import InstanceQueryInitializer
    from groundingdino.hierarchical.hierarchy import build_hierarchical_labels
    print("✓ All imports successful!")
except ImportError as e:
    print(f"✗ Import error: {e}")

✓ All imports successful!


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/Agri-VG/GroundingDINO/groundingdino/models/GroundingDINO/utils.py:61: SyntaxWarning: invalid escape sequence '\s'
  - memory: bs, \sum{hw}, d_model
/content/Agri-VG/GroundingDINO/groundingdino/hierarchical/hierarchy.py:331: SyntaxWarning: invalid escape sequence '\{'
  L^pair = -log(exp(q_i · T_pos,i^l / τ) / Σ_{a∈A\{i}} exp(q_i · f_a / τ))


In [10]:
zip_path = "/content/drive/MyDrive/images.zip"

# Step 3: Folder to extract the contents
extract_to = "/content/data/images"
os.makedirs(extract_to, exist_ok=True)

# Step 4: Unzip the file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("✅ Files extracted to:", os.path.abspath(extract_to))


✅ Files extracted to: /content/data/images


In [11]:
import json
import os

# Load and verify annotations
print("Checking annotation files...")

# Load grefs.json
with open('/content/Agri-VG/grefs(unc).json', 'r') as f:
    grefs = json.load(f)
print(f"✓ grefs.json loaded: {len(grefs['images'])} images")

# Load instances.json
with open('/content/Agri-VG/instances.json', 'r') as f:
    instances = json.load(f)
print(f"✓ instances.json loaded: {len(grefs['images'])} images, {len(instances['annotations'])} annotations")

# Check categories
categories = instances.get('categories', [])
print(f"✓ Categories: {[cat['name'] for cat in categories]}")

# Verify image files exist
sample_image = grefs['images'][0]
img_path = f"/content/data/images/images/{sample_image['file_name']}"
print(f"✓ Sample image exists: {os.path.exists(img_path)}")

Checking annotation files...
✓ grefs.json loaded: 8034 images
✓ instances.json loaded: 8034 images, 78288 annotations
✓ Categories: ['crop', 'weed']
✓ Sample image exists: True


In [22]:
config = {
    # Data paths
    "data_dir": "/content/data/images/images",
    "grefs_json": "/content/Agri-VG/grefs(unc).json",
    "instances_json": "/content/Agri-VG/instances.json",

    # Model paths
    "config_file": "/content/Agri-VG/GroundingDINO/groundingdino/config/GroundingDINO_SwinB_cfg.py",
    "checkpoint_path": "/content/weights/groundingdino_swinb_cogcoor.pth",

    # Training hyperparameters
    "epochs": 10,
    "batch_size": 12,  # Adjust based on GPU memory (T4: 3-6, A100: 12-24)
    "learning_rate": 0.001,
    "lr_decay_epochs": "3,6",
    "lr_decay_rate": 0.1,

    # Loss weights
    "lambda_hmlc": 1.0,
    "lambda_l1": 1.0,
    "lambda_giou": 1.0,
    "temp": 0.03,
    "loss_type": "hmce",

    # Optimization
    "weight_decay": 1e-4,
    "momentum": 0.9,

    # Logging
    "save_folder": "/content/checkpoints",
    "tb_folder": "/content/tensorboard",
    "save_freq": 10,
    "print_freq": 10,

    # Resources
    "workers": 4,
    "gpu": 0,
}

# Create output directories
os.makedirs(config["save_folder"], exist_ok=True)
os.makedirs(config["tb_folder"], exist_ok=True)

print("Training configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

Training configuration:
  data_dir: /content/data/images/images
  grefs_json: /content/Agri-VG/grefs(unc).json
  instances_json: /content/Agri-VG/instances.json
  config_file: /content/Agri-VG/GroundingDINO/groundingdino/config/GroundingDINO_SwinB_cfg.py
  checkpoint_path: /content/weights/groundingdino_swinb_cogcoor.pth
  epochs: 10
  batch_size: 12
  learning_rate: 0.001
  lr_decay_epochs: 3,6
  lr_decay_rate: 0.1
  lambda_hmlc: 1.0
  lambda_l1: 1.0
  lambda_giou: 1.0
  temp: 0.03
  loss_type: hmce
  weight_decay: 0.0001
  momentum: 0.9
  save_folder: /content/checkpoints
  tb_folder: /content/tensorboard
  save_freq: 10
  print_freq: 10
  workers: 4
  gpu: 0


In [23]:
cmd = f"""
python train_hierarchical_contrastive.py \
    --data-dir {config['data_dir']} \
    --grefs-json "{config['grefs_json']}" \
    --instances-json {config['instances_json']} \
    --config-file {config['config_file']} \
    --checkpoint-path {config['checkpoint_path']} \
    --epochs {config['epochs']} \
    --batch-size {config['batch_size']} \
    --learning-rate {config['learning_rate']} \
    --lr-decay-epochs {config['lr_decay_epochs']} \
    --lr-decay-rate {config['lr_decay_rate']} \
    --lambda-hmlc {config['lambda_hmlc']} \
    --lambda-l1 {config['lambda_l1']} \
    --lambda-giou {config['lambda_giou']} \
    --temp {config['temp']} \
    --loss-type {config['loss_type']} \
    --weight-decay {config['weight_decay']} \
    --momentum {config['momentum']} \
    --save-folder {config['save_folder']} \
    --tb-folder {config['tb_folder']} \
    --save-freq {config['save_freq']} \
    --print-freq {config['print_freq']} \
    --workers {config['workers']} \
    --gpu {config['gpu']}
"""

print("Training command:")
print(cmd)

Training command:

python train_hierarchical_contrastive.py     --data-dir /content/data/images/images     --grefs-json "/content/Agri-VG/grefs(unc).json"     --instances-json /content/Agri-VG/instances.json     --config-file /content/Agri-VG/GroundingDINO/groundingdino/config/GroundingDINO_SwinB_cfg.py     --checkpoint-path /content/weights/groundingdino_swinb_cogcoor.pth     --epochs 10     --batch-size 12     --learning-rate 0.001     --lr-decay-epochs 3,6     --lr-decay-rate 0.1     --lambda-hmlc 1.0     --lambda-l1 1.0     --lambda-giou 1.0     --temp 0.03     --loss-type hmce     --weight-decay 0.0001     --momentum 0.9     --save-folder /content/checkpoints     --tb-folder /content/tensorboard     --save-freq 10     --print-freq 10     --workers 4     --gpu 0



In [14]:
!pip install tensorboardX

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.6 MB/s eta 0:00:00


In [26]:
!{cmd}

2025-10-29 03:48:52.482711: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761709732.694216   13165 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761709732.720180   13165 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761709732.853088   13165 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761709732.853139   13165 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761709732.853147   13165 computation_placer.cc:177] computation placer alr